In [0]:
!pip install --no-cache-dir -U pageindex openai python-dotenv

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from groq import Groq

In [0]:
%restart_python

In [0]:
from dotenv import load_dotenv
import os

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Groq key loaded:   ", "✅" if GROQ_API_KEY else "❌ Missing!")

PageIndex key loaded: ✅
Groq key loaded:    ✅


In [0]:
from pageindex import PageIndexClient
from groq import Groq

pi_client   = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ PageIndex client ready")
print("✅ Groq client ready")

✅ PageIndex client ready
✅ Groq client ready


In [0]:
PDF_PATH = "/Workspace/Users/ankita.ghosh1@genpact.com/Projects/Vectorless Rag/Pyramid pooling for Deep Convolutional Network.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: /Workspace/Users/ankita.ghosh1@genpact.com/Projects/Vectorless Rag/Pyramid pooling for Deep Convolutional Network.pdf
✅ Uploaded!
📋 Document ID: pi-cmo5x49c612dj01r44hwq0bhk
   (Save this ID — you'll use it throughout the notebook)


In [0]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.
import time
print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: completed

✅ Tree index ready!


In [0]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
import json
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Spatial Pyramid Pooling in Deep Convolutional Networks for Visual Recognition",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "This text introduces Spatial Pyramid Pooling (SPP) and SPP-net, a novel CNN architecture that eliminates the need for fixed-size inputs, improving recognition accuracy for arbitrary image sizes and scales. SPP-net is robust to object deformations and enhances CNN-based image classification, achieving state-of-the-art results on datasets like ImageNet 2012, Pascal VOC 2007, and Caltech101. It also significantly boosts object detection performance by enabling efficient feature pooling from the entire image, leading to substantial speedups over methods like R-CNN and achieving high rankings in the ILSVRC 2014 competition for both object detection and image classification.",
  "text": "# Spatial Pyramid Pooling in Deep Convolutional Networks for Visual Recognition\n\nKaiming He, Xiangyu Z

In [0]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Spatial Pyramid Pooling in Deep Convolutional Networks for Visual Recognition  (p.1)
  └─ [0001] 1 INTRODUCTION  (p.1)
  └─ [0002] 2 Deep Networks with Spatial Pyramid Pooling  (p.2)
    └─ [0003] 2.1 Convolutional Layers and Feature Maps  (p.2)
    └─ [0004] 2.2 The Spatial Pyramid Pooling Layer  (p.3)
    └─ [0005] 2.3 Training the Network  (p.4)
    └─ [0006] Single-size training  (p.4)
    └─ [0007] Multi-size training  (p.4)
  └─ [0008] 3 SPP-NET FOR IMAGE CLASSIFICATION  (p.5)
    └─ [0009] 3.1 Experiments on ImageNet 2012 Classification  (p.5)
      └─ [0010] 3.1.1 Baseline Network Architectures  (p.5)
      └─ [0011] 3.1.2 Multi-level Pooling Improves Accuracy  (p.5)
      └─ [0012] 3.1.3 Multi-size Training Improves Accuracy  (p.6)
      └─ [0013] 3.1.4 Full-image Representations Improve Accuracy  (p.6)
      └─ [0014] 3.1.5 Multi-view Testing on Feature Maps  (p.6)
      └─ [0015] 3.1.6 Summary and Results for ILSVRC 2014  (p.7)
    └─ [0016

In [0]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 26
   Each node = one retrievable section of the document


In [0]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-120b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [0]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the topic covered in the Training in Network?"
print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the topic covered in the Training in Network?

🧠 LLM Reasoning:
The query asks about the topic covered in the 'Training in Network' section. In the document tree, the section titled '2.3 Training the Network' (node_id 0005) directly addresses training the network, describing how the network can be trained with standard back‑propagation regardless of input size. The subsections 'Single-size training' (node_id 0006) and 'Multi-size training' (node_id 0007) further elaborate on specific training regimes. These three nodes are the most relevant for answering the query.

🎯 Selected Node IDs: ['0005', '0006', '0007']


In [0]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [0]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [0]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [0]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What is the topic covered in the Training in Network??",
    tree=pageindex_tree
)

🔍 Query: What is the topic covered in the Training in Network??

🧠 Reasoning: The query asks about the topic covered in the 'Training in Network' section. In the document tree, the section titled "2.3 Training the Network" (node_id 0005) directly matches this phrase. It discuss...
🎯 Retrieved node IDs: ['0005', '0006', '0007']
📄 Sections found: ['2.3 Training the Network', 'Single-size training', 'Multi-size training']

📝 Answer:
The **Training the Network** section discusses how to train the SPP‑net using GPU‑friendly, fixed‑size inputs while preserving spatial‑pyramid‑pooling behavior. It describes two training strategies:

* **Single‑size training** – training on a fixed‑size (224 × 224) cropped image and implementing the pyramid pooling as sliding‑window pooling (Section “2.3 Training the Network”, p. 4).  
* **Multi‑size training** – training on two fixed sizes (224 × 224 and 180 × 180) that share parameters, alternating epochs to simulate varying‑size inputs (Section “Multi‑size 

In [0]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What are the syllabus covered in that pdf?",
    "What are the syllabus covered in RAG?"
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What are the syllabus covered in that pdf?
A: **Syllabus of the PDF**

- **1 INTRODUCTION** – Page 1 (Section: ‘1 INTRODUCTION’)  
- **2 Deep Networks with Spatial Pyramid Pooling** – Page 2 (Section: ‘2 Deep Networks with Spatial Pyramid Pooling’)  
- **3 SPP‑NET FOR IMAGE CLASSIFICATION** – Page 5 (Section: ‘3 SPP‑NET FOR IMAGE CLASSIFICATION...
-------------------------------------------------------

Q: What are the syllabus covered in RAG?
A: ⚠️ No relevant sections found in the document....
-------------------------------------------------------


In [0]:
FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [0]:
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""

In [0]:
# ── Expert-guided tree search ────────────────────────────────────────────────

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "openai/gpt-oss-120b"
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """
    
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "page": n.get("page_index", "?"),
                     "summary": n.get("text", "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [0]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Details of The Spatial Pyramid Pooling Layer."

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:800])

🔍 Query: Details of The Spatial Pyramid Pooling Layer.

── Without Expert Rules ──
Nodes: ['0004', '0002']

── With Expert Rules ──
Nodes: ['0004']
Reasoning: The query asks for details of the Spatial Pyramid Pooling Layer. The document node specifically titled '2.2 The Spatial Pyramid Pooling Layer' (node_id 0004) will contain the detailed description. No other nodes focus on that layer, so the most relevant node is 0004.


In [0]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

# Run it
answer = expert_rag(
    query="Details of The Spatial Pyramid Pooling Layer.",
    tree=pageindex_tree,
    rules=FINANCIAL_EXPERT_RULES
)
print(answer)

The Spatial Pyramid Pooling (SPP) layer replaces the network’s final pooling layer (e.g., pool₅) so that the convolutional part can accept images of **any size or aspect ratio** (2.2 The Spatial Pyramid Pooling Layer, p. 3).  

* **Fixed‑dimensional output:** For each spatial bin the responses of every filter in the last convolutional layer are pooled (max‑pooling is used throughout the paper). If the last convolutional layer has *k* filters and the pyramid contains *M* bins, the SPP layer produces a *k × M*‑dimensional vector that feeds the fully‑connected layer (2.2 The Spatial Pyramid Pooling Layer, p. 3).  

* **Spatial bins:** The image is divided into bins whose sizes are **proportional to the image dimensions**, so the **number of bins (M) stays constant** regardless of the input size. This preserves spatial information compared with a plain Bag‑of‑Words pooling (2.2 The Spatial Pyramid Pooling Layer, p. 3).  

* **Multi‑scale handling:** Because the convolutional filters keep t

In [0]:
question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
Here are the key findings from the paper:

---

## Key Findings: SPP-net (Spatial Pyramid Pooling Networks)

### 1. 🔑 Solves the Fixed-Input Limitation of CNNs
Traditional CNNs require a fixed input size (e.g., 224×224), forcing images to be cropped or warped — causing content loss or distortion. SPP-net introduces a **Spatial Pyramid Pooling layer** between the convolutional and fully-connected layers, enabling the network to accept **images of any size or scale**.

### 2. 📈 Improved Image Classification Accuracy
- On **ImageNet 2012**, SPP-net achieved **9.08% top-5 error** (single model), outperforming prior methods like OverFeat and Zeiler & Fergus.
- Combined with 11 models: **8.06% top-5 error**, ranking **#3 in ILSVRC 2014** out of 38 teams.
- Achieves state-of-the-art on **Pascal VOC 2007** and a record **93.42% accuracy on Caltech101**.

### 3. ⚡ Massive Speedup in Object Detection (24–102× faster than R-CNN)
Instead of running convolutions on each region pr

In [0]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history
    
    conversation_history.append({"role": "user", "content": user_message})
    
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Simulate a 3-turn conversation
questions = [
    "What problem in traditional CNNs does Spatial Pyramid Pooling (SPP) aim to solve?",
    "What is the main idea behind SPP-net architecture?",
    "How does spatial pyramid pooling generate a fixed-length output from variable-sized inputs?"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)


👤 User: What problem in traditional CNNs does Spatial Pyramid Pooling (SPP) aim to solve?
🤖 Assistant: SPP addresses a fundamental limitation of traditional CNNs: **the requirement for a fixed-size input image** (e.g., 224×224). Here's a breakdown of the problem and why it matters:

### The Core Problem
Traditional CNNs impose a fixed input size not because of the convolutional layers — which can actually handle images of any size — but because of the **fully-connected layers** at the end of the ne...
-------------------------------------------------------

👤 User: What is the main idea behind SPP-net architecture?
🤖 Assistant: Here's a clear breakdown of the main idea behind SPP-net:

---

### SPP-net Architecture: The Core Idea

SPP-net takes a standard CNN and makes **one key structural change**: it replaces the last pooling layer (e.g., `pool5`) with a **Spatial Pyramid Pooling (SPP) layer**, inserted between the final convolutional layer and the fully-connected layers.

#### How 

In [0]:
!ls

 PageIndex					       Vectorless_RAG.ipynb
'Pyramid pooling for Deep Convolutional Network.pdf'


In [0]:
!ls logs

ls: cannot access 'logs': No such file or directory


In [0]:
!find . -name "*.json"

./PageIndex/logs/Pyramid pooling for Deep Convolutional Network.pdf_20260419_110236.json
./PageIndex/examples/documents/results/2023-annual-report-truncated_structure.json
./PageIndex/examples/documents/results/2023-annual-report_structure.json
./PageIndex/examples/documents/results/PRML_structure.json
./PageIndex/examples/documents/results/Regulation Best Interest_Interpretive release_structure.json
./PageIndex/examples/documents/results/Regulation Best Interest_proposed rule_structure.json
./PageIndex/examples/documents/results/earthmover_structure.json
./PageIndex/examples/documents/results/four-lectures_structure.json
./PageIndex/examples/documents/results/q1-fy25-earnings_structure.json
./PageIndex/examples/workspace/12345678-abcd-4321-abcd-123456789abc.json
./PageIndex/examples/workspace/_meta.json
